# 07 | Optional independent spectroscopic comparison (LAMOST DR11)
**Purpose.** Compare the original 7,000-source balanced held-out Gaia test predictions with the independently derived LAMOST spectroscopic classes on matched sources.

**This step is optional for code reproducibility.** It requires an existing `data/lamost_dr11_matches.csv` file or live access to an astronomical TAP service. The dataset is not packaged here. By default, remote access is disabled in `config/experiment.json`; enabling it may take substantial time, fail, or return updated data.

**Interpretation:** the LAMOST cross-match covers only its observed overlap and is not an unbiased validation of the complete Galactic population.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd().resolve()
if not (ROOT / "config" / "experiment.json").exists():
    ROOT = ROOT.parent
if not (ROOT / "config" / "experiment.json").exists():
    raise FileNotFoundError("Start Jupyter from the repository root or notebooks/ folder.")
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
from IPython.display import display
from src.gaia_pipeline import *
cfg = load_config(ROOT)
np.random.seed(cfg["random_state"])
print("Repository:", ROOT)
import re
import hashlib
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, f1_score, matthews_corrcoef, classification_report, confusion_matrix
balanced = read_object(ROOT, "preprocessed.joblib")["balanced"]
benchmark = read_object(ROOT, "benchmark.joblib")
ensembles = read_object(ROOT, "ensembles.joblib")
classes = benchmark["classes"]
test_idx = benchmark["test_idx"]
original = benchmark["by_feature"]["combined_features"]
external_test = balanced.iloc[test_idx][["Source", "RA_ICRS", "DE_ICRS", "SpType-ELS"]].copy().reset_index(drop=True)
external_test["GaiaClass"] = external_test["SpType-ELS"].astype(str)
external_test["CatBoostPrediction"] = np.asarray(classes)[original["predictions"]["CatBoost"]]
external_test["EnsemblePrediction"] = np.asarray(classes)[ensembles["predictions"][ENSEMBLE_NAMES["hard_voting"]]]
external_test["gaia_id"] = external_test["Source"].astype("string")
assert external_test["Source"].is_unique
print("Held-out Gaia sources for optional cross-matching:", len(external_test))

## Load archived matches or query TAP (opt-in)
The source notebook used LAMOST DR11 `public.catalogue` and a VizieR alternative. Gaia IDs must retain exact 64-bit precision; the official LAMOST ADQL query treats them as quoted strings. Retrieved batches are cached by source-ID hash and service to support restart. Set `run_external_query=true` only if no local match file exists and network access is permitted.

In [ ]:
matches_path = ROOT / cfg["lamost_match_csv"]
lamost_raw = None
if matches_path.exists():
    lamost_raw = pd.read_csv(matches_path, dtype={"gaia_id": "string"})
    print("Loaded archived LAMOST matches:", len(lamost_raw))
elif cfg["run_external_query"]:
    import pyvo
    import requests
    services = [
        ("LAMOST DR11 official", "https://www.lamost.org/dr11/v2.0/voservice/tap"),
        ("VizieR DR11", "https://tapvizier.cds.unistra.fr/TAPVizieR/tap")
    ]
    usable = None
    for name, endpoint in services:
        try:
            response = requests.get(endpoint + "/capabilities", timeout=(5, 12))
            response.raise_for_status()
            usable = (name, endpoint)
            break
        except requests.exceptions.RequestException as exc:
            print(f"{name} unreachable: {type(exc).__name__}")
    if usable is None:
        raise ConnectionError("No configured LAMOST TAP service is reachable. Provide a cached match CSV instead.")
    name, endpoint = usable
    tap = pyvo.dal.TAPService(endpoint)
    cache_dir = output_dir(ROOT) / "lamost_cache"; cache_dir.mkdir(exist_ok=True)
    ids = external_test["Source"].to_numpy(dtype=np.int64)
    retrieved = []
    batch_size = cfg["lamost_batch_size"]
    for start in range(0, len(ids), batch_size):
        batch = ids[start:start + batch_size]
        digest = hashlib.sha256(batch.tobytes()).hexdigest()[:12]
        cache = cache_dir / f"{name.replace(' ', '_')}_{start // batch_size + 1:03d}_{digest}.csv"
        if cache.is_file():
            chunk = pd.read_csv(cache, dtype={"gaia_id": "string"})
        else:
            if name.startswith("VizieR"):
                identifiers = ", ".join(str(int(x)) for x in batch)
                sql = f"""SELECT l."GaiaDR3" AS gaia_id, l."Class" AS lamostobjectclass, l."subClass" AS lamostsubclass, l."snrg" AS snr_g, l."ObsID" AS lamostobsid FROM "V/162/dr11l" AS l WHERE l."GaiaDR3" IN ({identifiers}) AND l."Class" = 'STAR' AND l."snrg" >= {cfg["lamost_snr_min"]}"""
            else:
                identifiers = ", ".join(f"'{int(x)}'" for x in batch)
                sql = f"""SELECT l.gaia_source_id AS gaia_id, l."class" AS lamostobjectclass, l.subclass AS lamostsubclass, l.snrg AS snr_g, l.obsid AS lamostobsid FROM public.catalogue AS l WHERE l.gaia_source_id IN ({identifiers}) AND l."class" = 'STAR' AND l.snrg >= {cfg["lamost_snr_min"]}"""
            result = tap.run_async(sql, maxrec=10000, timeout=600)
            chunk = result.to_table().to_pandas()
            chunk.columns = chunk.columns.str.lower()
            if len(chunk) >= 10000:
                raise ValueError(f"Potentially truncated TAP batch {start // batch_size + 1}; reduce lamost_batch_size.")
            if not chunk.empty:
                if pd.api.types.is_float_dtype(chunk["gaia_id"]):
                    raise TypeError("LAMOST Gaia IDs were converted to float; precision is not reliable.")
                chunk["gaia_id"] = chunk["gaia_id"].astype("string")
            chunk.to_csv(cache, index=False)
        retrieved.append(chunk)
        print(f"TAP batch {start // batch_size + 1}: {len(chunk)} rows")
    lamost_raw = pd.concat(retrieved, ignore_index=True)
    matches_path.parent.mkdir(parents=True, exist_ok=True)
    lamost_raw.to_csv(matches_path, index=False)
    print("Saved LAMOST matches:", matches_path)
else:
    print("No archived LAMOST match file and live TAP access is disabled; external validation skipped.")

## Harmonize LAMOST classes and resolve repeated observations
Retain `STAR` records with G-band SNR at least 10, map spectral subclasses to the broad O/B/A/F/G/K/M groups, count sources with contradictory repeat classifications, and select the highest-SNR valid observation for each Gaia identifier.

In [ ]:
def map_lamost_class(value):
    if pd.isna(value):
        return np.nan
    label = re.sub(r"\s+", "", str(value).upper())
    match = re.match(r"^([OBAFGKM])\d", label)
    if match:
        return match.group(1)
    return "M" if re.match(r"^(?:G|D|SD|ESD|USD)M\d", label) else np.nan

lamost_validated, conflicting_ids = None, set()
if lamost_raw is not None and len(lamost_raw):
    required = {"gaia_id", "lamostobjectclass", "lamostsubclass", "snr_g"}
    if required - set(lamost_raw.columns):
        raise ValueError(f"LAMOST CSV lacks expected columns: {sorted(required - set(lamost_raw.columns))}")
    raw = lamost_raw.copy()
    raw["gaia_id"] = raw["gaia_id"].astype("string").str.strip()
    raw["LAMOSTClass"] = raw["lamostsubclass"].apply(map_lamost_class)
    raw["SNR_g"] = pd.to_numeric(raw["snr_g"], errors="coerce")
    raw = raw[(raw["lamostobjectclass"].astype(str).str.upper().str.strip() == "STAR") & raw["LAMOSTClass"].isin(cfg["spectral_classes"]) & (raw["SNR_g"] >= cfg["lamost_snr_min"])]
    disagreement = raw.groupby("gaia_id")["LAMOSTClass"].nunique()
    conflicting_ids = set(disagreement[disagreement > 1].index)
    lamost_validated = raw.sort_values("SNR_g", ascending=False).drop_duplicates("gaia_id").merge(external_test, on="gaia_id", how="inner", validate="one_to_one")
    print("Matched Gaia held-out sources:", len(lamost_validated), "| Conflicting repeat classes:", len(conflicting_ids))
    display(lamost_validated["LAMOSTClass"].value_counts().reindex(cfg["spectral_classes"], fill_value=0))
else:
    print("No validated LAMOST matches to evaluate.")

## Compare both label systems on identical matched sources
Macro-F1 is reported over all seven classes and, separately, classes represented by LAMOST. The matched sample is selective and the LAMOST pipeline itself may contain classification error. Conflicting repeat observations are also excluded in a sensitivity check.

In [ ]:
if lamost_validated is not None and len(lamost_validated):
    rows = []
    for subset_name, subset in [("All matches", lamost_validated), ("Consistent repeats", lamost_validated[~lamost_validated["gaia_id"].isin(conflicting_ids)])]:
        supported = [cls for cls in cfg["spectral_classes"] if (subset["LAMOSTClass"] == cls).any()]
        for model_name, prediction_col in [("CatBoost", "CatBoostPrediction"), ("Hard Voting", "EnsemblePrediction")]:
            for reference_name, target_col in [("Gaia labels", "GaiaClass"), ("LAMOST labels", "LAMOSTClass")]:
                true, pred = subset[target_col].to_numpy(), subset[prediction_col].to_numpy()
                rows.append({"Subset": subset_name, "Model": model_name, "Reference": reference_name, "Sources": len(subset), "Accuracy": accuracy_score(true, pred), "Macro-F1 (7 classes)": f1_score(true, pred, labels=cfg["spectral_classes"], average="macro", zero_division=0), "Macro-F1 (LAMOST-supported)": f1_score(true, pred, labels=supported, average="macro", zero_division=0), "MCC": matthews_corrcoef(true, pred)})
    external_table = pd.DataFrame(rows)
    save_table(external_table, ROOT, "lamost_external_metrics.csv")
    display(external_table.round(4))
    FIG = output_dir(ROOT) / "figures"; FIG.mkdir(exist_ok=True)
    for name, column in [("CatBoost", "CatBoostPrediction"), ("Hard Voting", "EnsemblePrediction")]:
        matrix = confusion_matrix(lamost_validated["LAMOSTClass"], lamost_validated[column], labels=cfg["spectral_classes"])
        fig, ax = plt.subplots(figsize=(7, 5))
        sns.heatmap(matrix, annot=True, fmt="d", cmap="viridis", xticklabels=cfg["spectral_classes"], yticklabels=cfg["spectral_classes"], ax=ax)
        ax.set(xlabel="Predicted spectral class", ylabel="LAMOST spectral class", title=f"External LAMOST comparison: {name}")
        fig.tight_layout(); fig.savefig(FIG / f"lamost_confusion_{name.replace(' ', '_')}.png"); plt.show()
else:
    print("External metrics not created because no LAMOST observations were available.")